# NumPy Problem Set — Plate Reader & Chromatography

Fifteen problems covering array creation, indexing, reshaping, broadcasting,
aggregation, sorting, joining, and boolean masking.

Each problem has a prompt, an empty code cell, and a test cell that checks your
answer with `assert` statements. A test cell that prints nothing but a pass
message means you got it right; a failing `assert` tells you what went wrong.

**Two working rules, carried over from the NumPy module:**

1. Several cells reassign a variable to a modified version of itself. Those are
   not safe to re-run. If anything looks doubled or mis-shaped, restart the
   kernel and run every cell in order.
2. Predict the answer before you run it. The tests check your output, but only
   you can check your reasoning.

---

## The data

- `plate` — an 8 x 12 array of absorbance readings from a 96-well plate.
  Rows are labelled **A** through **H** top to bottom, columns **1** through **12**
  left to right.
- `trace` — a 24-point absorbance trace from a chromatography run, in time order.
- `peak_heights` — six detected peak heights from that trace.
- `gains` — a per-row detector gain correction factor, one per plate row.
- `sample_ids` — the sample ID assigned to each plate row.

Run the setup cell before anything else.

In [1]:
import numpy as np

np.random.seed(42)
plate = np.round(np.random.uniform(0.05, 2.50, size=(8, 12)), 3)

np.random.seed(7)
trace = np.round(np.random.uniform(0.00, 1.80, size=24), 3)

peak_heights = np.array([0.42, 1.87, 0.95, 2.31, 0.11, 1.40])
gains = np.array([1.00, 1.05, 1.10, 1.15, 1.20, 1.25, 1.30, 1.35])
sample_ids = np.arange(1, 9)

print("plate ", plate.shape)
print("trace ", trace.shape)
print("peak_heights", peak_heights.shape)
print("gains ", gains.shape)
print("sample_ids ", sample_ids.shape)

plate  (8, 12)
trace  (24,)
peak_heights (6,)
gains  (8,)
sample_ids  (8,)


---
## Challenge 01 — Creating arrays

### 1.1 Empty baseline [1 point]

Create a one-dimensional array called `baseline` holding one `0.0` for every
point in `trace`, to be filled in later with a fitted baseline.

Call the NumPy function built for this rather than typing the values out, and
make sure the result holds floats, not integers.

In [2]:
# YOUR CODE HERE
baseline = np.zeros(trace.shape[0], dtype=float)
print(baseline)

[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [3]:
assert isinstance(baseline, np.ndarray), "baseline must be a NumPy array"
assert baseline.ndim == 1, f"baseline must be 1-D, got {baseline.ndim}-D"
assert baseline.shape == (24,), f"expected shape (24,), got {baseline.shape}"
assert np.all(baseline == 0), "every element must be zero"
assert baseline.dtype == np.float64, f"expected float64, got {baseline.dtype} - pass a dtype"
print("1.1 passed")

1.1 passed


### 1.2 Blank plate [1 point]

Every well on this instrument carries a constant blank absorbance of `0.085`.

Build an array called `blank_plate` with the same shape as `plate`, every well
holding `0.085`. Derive the shape from `plate` rather than typing `(8, 12)`
by hand.

In [4]:
# YOUR CODE HERE
blank_plate = np.full(plate.shape, 0.085)
print(blank_plate)

[[0.085 0.085 0.085 0.085 0.085 0.085 0.085 0.085 0.085 0.085 0.085 0.085]
 [0.085 0.085 0.085 0.085 0.085 0.085 0.085 0.085 0.085 0.085 0.085 0.085]
 [0.085 0.085 0.085 0.085 0.085 0.085 0.085 0.085 0.085 0.085 0.085 0.085]
 [0.085 0.085 0.085 0.085 0.085 0.085 0.085 0.085 0.085 0.085 0.085 0.085]
 [0.085 0.085 0.085 0.085 0.085 0.085 0.085 0.085 0.085 0.085 0.085 0.085]
 [0.085 0.085 0.085 0.085 0.085 0.085 0.085 0.085 0.085 0.085 0.085 0.085]
 [0.085 0.085 0.085 0.085 0.085 0.085 0.085 0.085 0.085 0.085 0.085 0.085]
 [0.085 0.085 0.085 0.085 0.085 0.085 0.085 0.085 0.085 0.085 0.085 0.085]]


In [5]:
assert isinstance(blank_plate, np.ndarray), "blank_plate must be a NumPy array"
assert blank_plate.shape == (8, 12), f"expected (8, 12), got {blank_plate.shape}"
assert np.all(blank_plate == 0.085), "every well must hold exactly 0.085"
print("1.2 passed")

1.2 passed


---
## Challenge 02 — Indexing and slicing

### 2.1 Read one well [1 point]

Plate rows are labelled **A-H** top to bottom and columns **1-12** left to right,
so well **A1** is the top-left corner.

Save the absorbance at well **C7** to a variable called `well_value`.

`well_value` must be a single number, not an array.

In [6]:
# YOUR CODE HERE
well_value = plate[2, 6]
print(well_value)

1.538


In [7]:
assert np.ndim(well_value) == 0, f"well_value must be a single number, got shape {np.shape(well_value)}"
assert np.isclose(well_value, 1.538), (
    f"got {well_value} - check how the plate labels map onto NumPy indices")
print("2.1 passed")

2.1 passed


### 2.2 Downsample the trace [1 point]

The detector logged `trace` at three times the rate you need.

Save every third reading, **starting from the first**, to a variable called
`every_third`.

In [8]:
# YOUR CODE HERE
every_third = trace[::3]
print(every_third)

[0.137 1.302 0.902 0.9   0.686 1.637 1.676 1.71 ]


In [9]:
expected = np.array([0.137, 1.302, 0.902, 0.9, 0.686, 1.637, 1.676, 1.71])
assert every_third.shape == (8,), f"expected shape (8,), got {every_third.shape}"
assert np.isclose(every_third[0], trace[0]), "must start at the FIRST reading"
assert np.allclose(every_third, expected), "wrong start point or wrong step size"
print("2.2 passed")

2.2 passed


### 2.3 Last row [1 point]

Save the entire bottom row of `plate` (row **H**) to a variable called `last_row`.

Write it so the code would still grab the bottom row if the plate gained or lost
rows — don't hardcode which row number is last.

In [10]:
# YOUR CODE HERE
last_row = plate[-1]
print(last_row)

[0.812 0.847 1.838 1.612 2.224 1.207 0.343 1.797 1.914 1.425 1.939 1.26 ]


In [11]:
assert last_row.shape == (12,), f"expected shape (12,), got {last_row.shape}"
assert np.allclose(last_row, plate[7]), "that is not the bottom row"
print("2.3 passed")

2.3 passed


---
## Challenge 03 — Reshaping

### 3.1 Fold the trace [1 point]

The 24-point `trace` was collected as four consecutive sweeps of six readings each.

Reshape it into a 4 x 6 array called `trace_grid`, one sweep per row, keeping the
readings in their original order.

Let NumPy work out the second dimension for you.

In [12]:
# YOUR CODE HERE
trace_grid = trace.reshape(4, -1)
print(trace_grid)

[[0.137 1.404 0.789 1.302 1.76  0.969]
 [0.902 0.13  0.483 0.9   1.223 1.447]
 [0.686 0.119 0.519 1.637 0.384 0.814]
 [1.676 0.045 1.081 1.71  0.415 0.987]]


In [13]:
assert trace_grid.shape == (4, 6), f"expected (4, 6), got {trace_grid.shape}"
assert np.isclose(trace_grid[0, 0], trace[0]), "the first reading should land at [0, 0]"
assert np.allclose(trace_grid[0], trace[:6]), "rows should fill in reading order"
print("3.1 passed")

3.1 passed


### 3.2 Unfold it again [1 point]

Flatten `trace_grid` back to one dimension, saving the result as `trace_flat`.

`trace_flat` must be an **independent copy**. Editing it must not reach back and
change `trace_grid`. One of the two ways you know of doing this guarantees that;
the other does not.

In [14]:
trace_flat = trace_grid.flatten()
print(trace_flat)

[0.137 1.404 0.789 1.302 1.76  0.969 0.902 0.13  0.483 0.9   1.223 1.447
 0.686 0.119 0.519 1.637 0.384 0.814 1.676 0.045 1.081 1.71  0.415 0.987]


In [15]:
assert trace_flat.ndim == 1, f"trace_flat must be 1-D, got {trace_flat.ndim}-D"
assert trace_flat.shape == (24,), f"expected shape (24,), got {trace_flat.shape}"

_probe = trace_flat[0]
trace_flat[0] = -999.0
_is_view = bool(trace_grid[0, 0] == -999.0)
trace_flat[0] = _probe          # restores trace_grid too, if it was a view

assert not _is_view, "trace_flat is a VIEW of trace_grid, not a copy"
print("3.2 passed")

3.2 passed


---
## Challenge 04 — Broadcasting

### 4.1 Blank correction [1 point]

Subtract the blank value `0.085` from every well of `plate`, saving the result as
`plate_corrected`.

Do it with broadcasting, in one expression, and leave `plate` itself untouched.

In [16]:
# YOUR CODE HERE
plate_corrected = plate - 0.085
print(plate_corrected)

[[ 0.883  2.294  1.758  1.432  0.347  0.347  0.107  2.087  1.438  1.7
   0.015  2.341]
 [ 2.004  0.485  0.41   0.414  0.71   1.251  1.023  0.679  1.464  0.307
   0.681  0.863]
 [ 1.082  1.889  0.454  1.225  1.416  0.079  1.453  0.383  0.124  2.29
   2.331  1.946]
 [ 0.711  0.204  1.641  1.043  0.264  1.178  0.049  2.193  0.599  1.588
   0.729  1.239]
 [ 1.304  0.418  2.34   1.864  2.267  2.157  1.43   2.224  0.182  0.445
   0.076  0.762]
 [ 0.917  0.63   1.995  0.839  0.653  1.295  0.31   1.93   0.148  2.383
   1.857  0.452]
 [-0.021  1.963  1.697  1.751  1.855  0.146  0.843  0.249  2.08   1.492
   0.776  0.121]
 [ 0.727  0.762  1.753  1.527  2.139  1.122  0.258  1.712  1.829  1.34
   1.854  1.175]]


In [17]:
assert plate_corrected.shape == (8, 12), f"expected (8, 12), got {plate_corrected.shape}"
assert np.isclose(plate_corrected[0, 0], 0.883), "blank not subtracted correctly"
assert np.allclose(plate_corrected, plate - 0.085), "every well must drop by exactly 0.085"
assert np.isclose(plate[0, 0], 0.968), "plate itself must be UNCHANGED"
print("4.1 passed")

4.1 passed


### 4.2 Per-row gain correction [1 point]

Each plate row was read by a different detector channel, so each row needs its
own gain factor. `gains` holds those eight factors, one per row, in row order.

Multiply `plate_corrected` by its gains and save the result as `plate_gained`.
Row A scales by `1.00`, row B by `1.05`, and so on down to row H at `1.35`.

`gains` has shape `(8,)`. Think about what that lines up against before you write
the multiplication — a bare `plate_corrected * gains` will not do what you want.

In [18]:
# YOUR CODE HERE
gains = gains.reshape(-1, 1)
plate_gained = plate_corrected * gains
print(plate_gained)


[[ 0.883    2.294    1.758    1.432    0.347    0.347    0.107    2.087
   1.438    1.7      0.015    2.341  ]
 [ 2.1042   0.50925  0.4305   0.4347   0.7455   1.31355  1.07415  0.71295
   1.5372   0.32235  0.71505  0.90615]
 [ 1.1902   2.0779   0.4994   1.3475   1.5576   0.0869   1.5983   0.4213
   0.1364   2.519    2.5641   2.1406 ]
 [ 0.81765  0.2346   1.88715  1.19945  0.3036   1.3547   0.05635  2.52195
   0.68885  1.8262   0.83835  1.42485]
 [ 1.5648   0.5016   2.808    2.2368   2.7204   2.5884   1.716    2.6688
   0.2184   0.534    0.0912   0.9144 ]
 [ 1.14625  0.7875   2.49375  1.04875  0.81625  1.61875  0.3875   2.4125
   0.185    2.97875  2.32125  0.565  ]
 [-0.0273   2.5519   2.2061   2.2763   2.4115   0.1898   1.0959   0.3237
   2.704    1.9396   1.0088   0.1573 ]
 [ 0.98145  1.0287   2.36655  2.06145  2.88765  1.5147   0.3483   2.3112
   2.46915  1.809    2.5029   1.58625]]


In [19]:
assert plate_gained.shape == (8, 12), f"expected (8, 12), got {plate_gained.shape}"
assert np.allclose(plate_gained[0], plate_corrected[0] * 1.00), "row A should scale by 1.00"
assert np.allclose(plate_gained[7], plate_corrected[7] * 1.35), "row H should scale by 1.35"
assert not np.allclose(plate_gained[1], plate_corrected[1] * 1.00), (
    "gains are being applied down the COLUMNS - they belong to the rows")
print("4.2 passed")

4.2 passed


---
## Challenge 05 — Aggregating

### 5.1 Column means [1 point]

Each column of the plate holds one dilution level.

Compute the mean absorbance of each column of `plate_gained`, saving the result
as `col_means`. You should end up with one value per column.

In [20]:
# YOUR CODE HERE
col_means = np.mean(plate_gained, axis=0)
print(col_means)

[1.08253125 1.24818125 1.80618125 1.50461875 1.4736875  1.126725
 0.7979375  1.682425   1.172125   1.7036125  1.25708125 1.25444375]


In [21]:
assert col_means.shape == (12,), (
    f"expected shape (12,), got {np.shape(col_means)} - one value per COLUMN")
assert np.isclose(col_means[0], plate_gained[:, 0].mean()), "column A values averaged wrong"
assert np.allclose(col_means, plate_gained.mean(axis=0)), "check which axis you collapsed"
print("5.1 passed")

5.1 passed


---
## Challenge 06 — Sorting

### 6.1 Rank the peaks [1 point]

Sort `peak_heights` from largest to smallest and save the result as `peaks_desc`.

`peak_heights` must be left **unchanged** — the original detection order still
matters downstream. Pick the form of sorting that returns a new array rather than
rearranging the one you pass it.

In [22]:
# YOUR CODE HERE
peak_sorted = np.sort(peak_heights)
peaks_desc = np.flip(peak_sorted)
print(peaks_desc)

[2.31 1.87 1.4  0.95 0.42 0.11]


In [23]:
assert peaks_desc is not None, (
    "peaks_desc is None - the in-place .sort() method returns nothing")
assert peaks_desc.shape == (6,), f"expected shape (6,), got {peaks_desc.shape}"
assert np.all(np.diff(peaks_desc) <= 0), "not in descending order"
assert np.isclose(peaks_desc[0], 2.31), "the tallest peak should come first"
assert np.allclose(peak_heights, [0.42, 1.87, 0.95, 2.31, 0.11, 1.40]), (
    "peak_heights was modified - you used the destructive form")
print("6.1 passed")

6.1 passed


### 6.2 Sort each sweep in place [1 point]

Now sort the readings **within each sweep** of `trace_grid` into ascending order,
modifying `trace_grid` itself rather than building a new array.

This is the opposite instruction from 6.1, so it needs the opposite form.

In [24]:
# YOUR CODE HERE
trace_grid.sort(axis=1)
print(trace_grid)

[[0.137 0.789 0.969 1.302 1.404 1.76 ]
 [0.13  0.483 0.9   0.902 1.223 1.447]
 [0.119 0.384 0.519 0.686 0.814 1.637]
 [0.045 0.415 0.987 1.081 1.676 1.71 ]]


In [25]:
assert trace_grid.shape == (4, 6), f"shape must stay (4, 6), got {trace_grid.shape}"
assert np.all(np.diff(trace_grid, axis=1) >= 0), "each ROW must be ascending"
assert np.allclose(np.sort(trace_grid.flatten()), np.sort(trace)), (
    "the values changed - sorting should only reorder them")
print("6.2 passed")
print("trace is now:", trace[:6], "...")

6.2 passed
trace is now: [0.137 0.789 0.969 1.302 1.404 1.76 ] ...


---
## Challenge 07 — Joining

### 7.1 Attach sample IDs [1 point]

`sample_ids` holds the sample ID for each plate row.

Attach it to `plate_gained` as a new **final column**, saving the result as
`plate_labeled`. The twelve reading columns stay where they are and the IDs
become column thirteen.

`sample_ids` has shape `(8,)`. Check whether that can be joined on as-is.

In [26]:
# YOUR CODE HERE
sample_ids_2D = sample_ids.reshape(-1, 1)
plate_labeled = np.hstack((plate_gained, sample_ids_2D))
print(plate_labeled)

[[ 0.883    2.294    1.758    1.432    0.347    0.347    0.107    2.087
   1.438    1.7      0.015    2.341    1.     ]
 [ 2.1042   0.50925  0.4305   0.4347   0.7455   1.31355  1.07415  0.71295
   1.5372   0.32235  0.71505  0.90615  2.     ]
 [ 1.1902   2.0779   0.4994   1.3475   1.5576   0.0869   1.5983   0.4213
   0.1364   2.519    2.5641   2.1406   3.     ]
 [ 0.81765  0.2346   1.88715  1.19945  0.3036   1.3547   0.05635  2.52195
   0.68885  1.8262   0.83835  1.42485  4.     ]
 [ 1.5648   0.5016   2.808    2.2368   2.7204   2.5884   1.716    2.6688
   0.2184   0.534    0.0912   0.9144   5.     ]
 [ 1.14625  0.7875   2.49375  1.04875  0.81625  1.61875  0.3875   2.4125
   0.185    2.97875  2.32125  0.565    6.     ]
 [-0.0273   2.5519   2.2061   2.2763   2.4115   0.1898   1.0959   0.3237
   2.704    1.9396   1.0088   0.1573   7.     ]
 [ 0.98145  1.0287   2.36655  2.06145  2.88765  1.5147   0.3483   2.3112
   2.46915  1.809    2.5029   1.58625  8.     ]]


In [27]:
assert plate_labeled.shape == (8, 13), f"expected (8, 13), got {plate_labeled.shape}"
assert np.allclose(plate_labeled[:, -1], np.arange(1, 9)), (
    "sample IDs must be the LAST column, in row order")
assert np.allclose(plate_labeled[:, :12], plate_gained), "the readings must be preserved"
print("7.1 passed")

7.1 passed


---
## Challenge 08 — Boolean masking

### 8.1 Readings in the linear range [1 point]

The detector is only trustworthy between `0.5` and `1.5` absorbance units,
inclusive at both ends.

Build a boolean array called `in_range_mask`, the same shape as `plate_gained`,
that is `True` wherever a reading falls inside that band.

Then use it to pull those readings out of `plate_gained` into an array called
`in_range`.

In [28]:
# YOUR CODE HERE
in_range_mask = (plate_gained >= 0.5) & (plate_gained <= 1.5)
in_range = plate_gained[in_range_mask]
print(in_range)

[0.883   1.432   1.438   0.50925 0.7455  1.31355 1.07415 0.71295 0.71505
 0.90615 1.1902  1.3475  0.81765 1.19945 1.3547  0.68885 0.83835 1.42485
 0.5016  0.534   0.9144  1.14625 0.7875  1.04875 0.81625 0.565   1.0959
 1.0088  0.98145 1.0287 ]


In [29]:
assert in_range_mask.shape == (8, 12), f"mask must be (8, 12), got {in_range_mask.shape}"
assert in_range_mask.dtype == bool, f"mask must be boolean, got {in_range_mask.dtype}"
assert in_range.ndim == 1, "masking a 2-D array returns a 1-D array of the matches"
assert np.all(in_range >= 0.5), "a reading below 0.5 leaked through"
assert np.all(in_range <= 1.5), "a reading above 1.5 leaked through"
assert in_range.size == 30, (
    f"expected 30 readings in range, got {in_range.size}")
assert in_range.size == in_range_mask.sum(), "mask and selection disagree"
print("8.1 passed")

8.1 passed


### 8.2 Flag the saturated wells [1 point]

Any raw reading of `2.0` or above means the detector saturated and the number is
meaningless.

Build an array called `plate_flagged` that holds the raw `plate` readings with
every saturated well replaced by `np.nan`.

`plate` itself must come out of this unchanged, so think about what you are
assigning into.

In [30]:
# YOUR CODE HERE
plate_flagged = plate.copy()
plate_flagged[plate_flagged >= 2.0] = np.nan
print(plate_flagged)

[[0.968   nan 1.843 1.517 0.432 0.432 0.192   nan 1.523 1.785 0.1     nan]
 [  nan 0.57  0.495 0.499 0.795 1.336 1.108 0.764 1.549 0.392 0.766 0.948]
 [1.167 1.974 0.539 1.31  1.501 0.164 1.538 0.468 0.209   nan   nan   nan]
 [0.796 0.289 1.726 1.128 0.349 1.263 0.134   nan 0.684 1.673 0.814 1.324]
 [1.389 0.503   nan 1.949   nan   nan 1.515   nan 0.267 0.53  0.161 0.847]
 [1.002 0.715   nan 0.924 0.738 1.38  0.395   nan 0.233   nan 1.942 0.537]
 [0.064   nan 1.782 1.836 1.94  0.231 0.928 0.334   nan 1.577 0.861 0.206]
 [0.812 0.847 1.838 1.612   nan 1.207 0.343 1.797 1.914 1.425 1.939 1.26 ]]


In [31]:
assert plate_flagged.shape == (8, 12), f"expected (8, 12), got {plate_flagged.shape}"
assert np.isnan(plate_flagged).sum() == 18, (
    f"expected 18 saturated wells, flagged {np.isnan(plate_flagged).sum()}")
assert np.all(plate_flagged[~np.isnan(plate_flagged)] < 2.0), "a saturated well survived"
assert np.allclose(plate_flagged[~np.isnan(plate_flagged)],
                   plate[plate < 2.0]), "the surviving wells were altered"
assert not np.isnan(plate).any(), (
    "plate itself was modified - you assigned into the original, not a copy")
print("8.2 passed")

8.2 passed
